# MixedSingleTaskGP: End-to-End Mixed-Variable Bayesian Optimization

`robotorchan.models.MixedSingleTaskGP`を使い、**連続変数 + nominal categorical variable**の探索を最大化問題として実行します。

## このNotebookで確認すること

- raw mixed inputと`cat_dims`の意味
- mixed surrogateの学習とカテゴリ別posterior
- surrogate semanticsとcandidate-space semanticsの分離
- BoTorch `optimize_acqf_mixed`による有効カテゴリ候補の生成
- candidate評価、データ追加、短いmixed BO loop

関連理論:

- [Mixed Variables](../../docs/theory/05_mixed_variables.md)
- [Bayesian Optimization](../../docs/theory/01_bayesian_optimization.md)
- [Improvement-based acquisition](../../docs/theory/acquisition/02_improvement.md)

```text
Raw mixed data -> Mixed GP -> Posterior -> Acquisition
                                      -> Mixed optimization -> Valid candidate
                                                            -> Evaluation -> repeat
```

カテゴリコード`0, 1, 2`は数値距離を表しません。モデル側では`cat_dims=[1]`、候補生成側では有効カテゴリの列挙を使い、両方のsemanticsを一致させます。

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim.optimize import optimize_acqf_mixed

from robotorchan.models import MixedSingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 連続変数とカテゴリ変数の違い

前のNotebook 01では、入力は連続値 $x\in[0,1]$ だけでした。ここでは、連続値 $x$ とカテゴリ $c\in\{0,1,2\}$ の組 $(x,c)$ を最適化します。

カテゴリ番号は**識別用のラベル**であり、通常は `0 < 1 < 2` という順序や、`0` と `1` の距離が `1` と `2` の距離に等しいことを意味しません。例えば `0=触媒A`、`1=触媒B`、`2=触媒C` と考えられます。数値として渡しても、GPには `cat_dims=[1]` でカテゴリ列を明示します。

目的は、観測コストを抑えながら**どのカテゴリを選ぶか**と**そのカテゴリで連続条件をどう設定するか**を同時に決めることです。

## 2. Problem setup

入力は2次元です。`X[..., 0]`は`[0, 1]`の連続変数、`X[..., 1]`は`{0, 1, 2}`のnominal categoryです。目的関数を最大化します。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    x = X[..., 0]
    category = X[..., 1]
    base = torch.sin(2 * torch.pi * x)
    category_effect = torch.where(
        category == 0,
        torch.zeros_like(x),
        torch.where(
            category == 1,
            0.45 + 0.15 * torch.cos(4 * torch.pi * x),
            -0.25 + 0.25 * x,
        ),
    )
    return (base + category_effect).unsqueeze(-1)


bounds = torch.tensor([[0.0, 0.0], [1.0, 2.0]])
fixed_features_list = [{1: 0.0}, {1: 1.0}, {1: 2.0}]

## 3. Training data

各カテゴリで同じ8個の連続条件を初期観測します。実務ではカテゴリごとの観測位置が同一である必要はありません。

In [ ]:
x_cont = torch.linspace(0.05, 0.95, 8)
train_X = torch.cat(
    [
        torch.stack([x_cont, torch.full_like(x_cont, category)], dim=-1)
        for category in (0.0, 1.0, 2.0)
    ],
    dim=0,
)
train_Y = objective(train_X)

print("train_X:", train_X.shape)
print("train_Y:", train_Y.shape)
print("categories:", train_X[:, 1].unique())

### `cat_dims` とカテゴリ用カーネル

通常の連続GPでは、入力の距離が近いほど出力も似ているという仮定をカーネルで表現します。一方、名義カテゴリでは番号の差をそのまま距離として扱うのは適切ではありません。

`MixedSingleTaskGP(..., cat_dims=[1])` は、2列目（Pythonの列番号1）をカテゴリとして扱う混合入力モデルです。カテゴリ間の類似性と連続変数に対する滑らかな変化を組み合わせて学習します。

ここでの `train_X` は `(24, 2)`、`train_Y` は `(24, 1)` です。`cat_dims` は**モデル側の入力解釈**を指定するものであり、最適化時に使えるカテゴリの集合を自動的に指定するものではありません。

## 4. Model construction

2列目を`cat_dims=[1]`として明示します。one-hotへ変換せずraw category codeを渡します。

In [ ]:
model = MixedSingleTaskGP(train_X=train_X, train_Y=train_Y, cat_dims=[1])

print(type(model).__name__)
print("supports_mll:", model.supports_mll)
print("raw_data_names:", model.raw_data_names)
print("raw_train_X shape:", model.raw_train_X.shape)
print("raw_train_Y shape:", model.raw_train_Y.shape)

## 5. Model fitting

In [ ]:
model.fit()
mll = model.make_mll()
model.eval()

print(type(mll).__name__)

### カテゴリ別Posteriorの見方

同じ連続値 $x$ でも、カテゴリ $c$ によって予測平均 $\mu(x,c)$ と予測分散 $\sigma^2(x,c)$ は異なります。各カテゴリの曲線を重ねると、**カテゴリによる出力の違い**と、各カテゴリ内での**連続条件の影響**を比較できます。

`posterior.mean` は予測平均、`posterior.mvn.confidence_region()` はGPの潜在関数に関する約95%のモデル上の区間です。区間は測定値の予測区間や、真の関数を必ず含む保証とは異なります。

## 6. Posterior by category

連続gridを共有し、カテゴリだけを固定してposteriorを比較します。

In [ ]:
grid = torch.linspace(0.0, 1.0, 201)
posterior_by_category = {}

with torch.no_grad():
    for category in (0.0, 1.0, 2.0):
        test_X = torch.stack([grid, torch.full_like(grid, category)], dim=-1)
        posterior = model.posterior(test_X)
        lower, upper = posterior.mvn.confidence_region()
        posterior_by_category[int(category)] = {
            "mean": posterior.mean.squeeze(-1),
            "lower": lower.squeeze(-1),
            "upper": upper.squeeze(-1),
        }

print("posterior categories:", tuple(posterior_by_category))

## 7. Posterior visualization

In [ ]:
plt.figure(figsize=(9, 5))
for category in (0, 1, 2):
    mask = train_X[:, 1] == category
    result = posterior_by_category[category]
    plt.scatter(train_X[mask, 0], train_Y[mask, 0], label=f"observed c={category}")
    plt.plot(grid, result["mean"], label=f"posterior c={category}")

plt.xlabel("continuous x")
plt.ylabel("y")
plt.legend()
plt.show()

### Mixed BOでも獲得関数の意味は同じ

現在の最大観測値を $f_{\mathrm{best}}$ とすると、改善量の期待値は

$$\operatorname{EI}(x,c)=\mathbb E\left[\max\{f(x,c)-f_{\mathrm{best}},0\}\mid\mathcal D\right]$$

です。`qLogExpectedImprovement` はこの改善の期待値に対応する数値安定な対数形式で、**連続条件とカテゴリの両方**を入力として評価します。獲得関数は次に調べる候補の有望さを示す指標であり、目的関数の測定値そのものではありません。

## 8. Acquisition function

現在の最大観測値を基準に`qLogExpectedImprovement`を構築します。単一出力なので明示的なObjective / PosteriorTransformは不要です。

In [ ]:
best_f = train_Y.max()
acqf = qLogExpectedImprovement(model=model, best_f=best_f)
print("best_f:", best_f.item())

### モデルの指定と候補の制約は別の責務

- `cat_dims=[1]`：GPが2列目を名義カテゴリとして学習するための指定。
- `fixed_features_list=[{1: 0.0}, {1: 1.0}, {1: 2.0}]`：候補生成時に2列目を許可カテゴリのいずれかへ固定する指定。

`bounds` を `[0,2]` に設定する**だけ**では、カテゴリ値が `0.3` などにならないことは保証されません。この例では、BoTorchの `optimize_acqf_mixed` が各固定カテゴリについて連続変数を最適化し、最良の候補を選びます。

カテゴリ数が少ないときはこの列挙方式が明快です。カテゴリの組合せが膨大な場合は、別の探索アルゴリズムや組合せ最適化手法が必要になります。

## 9. Mixed acquisition optimization

ここではカテゴリ組合せが3通りだけなので、BoTorch標準の`optimize_acqf_mixed`で各カテゴリを列挙し、連続次元を最適化します。これはsurrogateの`cat_dims`とは別の責務です。

In [ ]:
candidate, acq_value = optimize_acqf_mixed(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
    fixed_features_list=fixed_features_list,
)
candidate_Y = objective(candidate)

print("candidate:", candidate)
print("continuous value:", candidate[0, 0].item())
print("category:", int(candidate[0, 1].item()))
print("acquisition value:", acq_value)
print("objective(candidate):", candidate_Y)

## 10. Validate candidate semantics and update data

optimizerが返したカテゴリが許可集合に含まれることを確認してから観測を追加します。

In [ ]:
allowed_categories = torch.tensor([0.0, 1.0, 2.0])
assert torch.any(candidate[0, 1] == allowed_categories)

updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)
print("updated shapes:", updated_X.shape, updated_Y.shape)

### 実験データの更新と繰り返し

`optimize_acqf_mixed` が返す `next_X` は**提案された実験条件**です。`objective(next_X)` で得る `next_Y` は、その条件で実際に評価した結果に相当します。両者をデータに追加してGPを学習し直すことで、次回の候補が新しい観測を反映します。

このNotebookでは3回だけ繰り返します。カテゴリの許可集合の検証は `assert` で行いますが、実際の実験ではカテゴリ以外の装置制約・安全制約も別途考慮してください。

## 11. Short full mixed BO loop

3 iterationだけ、refit → acquisition → mixed candidate generation → evaluationを繰り返します。

In [ ]:
loop_X = train_X.clone()
loop_Y = train_Y.clone()
history = []

for iteration in range(3):
    loop_model = MixedSingleTaskGP(train_X=loop_X, train_Y=loop_Y, cat_dims=[1])
    loop_model.fit()
    loop_model.eval()

    loop_acqf = qLogExpectedImprovement(model=loop_model, best_f=loop_Y.max())
    next_X, next_acq_value = optimize_acqf_mixed(
        acq_function=loop_acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
        fixed_features_list=fixed_features_list,
    )
    next_Y = objective(next_X)
    assert torch.any(next_X[0, 1] == allowed_categories)
    history.append((next_X[0, 0].item(), int(next_X[0, 1].item()), next_Y.item()))
    loop_X = torch.cat([loop_X, next_X], dim=0)
    loop_Y = torch.cat([loop_Y, next_Y], dim=0)

for iteration, (x_value, category, y_value) in enumerate(history, start=1):
    print(f"iteration={iteration}: x={x_value:.4f}, category={category}, y={y_value:.4f}")

print("final best observed value:", loop_Y.max().item())

## 12. When to use this method

`MixedSingleTaskGP`は、温度 + 触媒種、圧力 + 装置種、組成比 + 製法など、連続条件とnominal categorical conditionが混在する探索に適します。

カテゴリ値が少なく列挙可能なら`optimize_acqf_mixed`は明快です。カテゴリ組合せが多い場合は、全列挙ではなくcandidate set、mixed-capable derivative-free optimizer、combinatorial strategyなどを検討します。

## 13. Limitations and caveats

- integer variableはnominal categoryと同一ではありません。
- task / fidelity / context / hierarchy列はdesign-side `cat_dims`と意味が異なります。
- mixed surrogateを使うだけでは有効candidateは保証されません。
- 既知の実行不可能条件はcandidate search space側で除外するのが基本です。
- この例は1カテゴリ次元・逐次・単一目的です。mixed + constraints / batch / multi-objectiveは専用workflowで扱います。

## 14. Related robotorchan components

robotorchanには`MixedVariableSpace`とmixed-capable optimizer backendもあります。ただし、この少数カテゴリ列挙の例では、理論章とBoTorch標準APIの対応を明確にするため`optimize_acqf_mixed`を直接使用します。

## 15. References

詳細な数理・探索方式・組合せ爆発の議論は冒頭のMixed Variables theoryを参照してください。